<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Laboratorios/03_Lab_naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## CARLOS MARIO GARCIA ALVAREZ. CC: 1036963351. LAB 3

# Laboratorio: Naive Bayes Bernoulli para detectar spam

## Objetivo
Construir desde cero un clasificador **Naive Bayes Bernoulli** a partir de un corpus pequeño de correos.

Al terminar debes poder explicar y calcular:

$
P(Y),\qquad P(X_i\mid Y),\qquad P(X\mid Y),\qquad P(Y\mid X)
$

y comprender por qué la hipótesis

$
X_i \perp X_j\mid Y
$

reduce drásticamente la complejidad del modelo.

## Contexto

Queremos clasificar un correo como

$
Y=1:\ \text{spam}, \qquad Y=0:\ \text{normal}.
$

Usaremos cinco palabras del vocabulario:


$V=\{\text{dinero, gratis, premio, proyecto, reunion}\}.$

Cada correo se representa por

$X=(X_1,\ldots,X_5),$

donde

$
X_i=\begin{cases}
1 & \text{si aparece la palabra }i,\\
0 & \text{si no aparece.}
\end{cases}
$
Por ejemplo, "dinero gratis" se representa como $(1,1,0,0,0).$


In [1]:
corpus = [
    ("spam",   "gana dinero gratis"),
    ("spam",   "dinero gratis ahora"),
    ("spam",   "premio dinero gratis"),
    ("spam",   "gana premio ahora"),
    ("spam",   "en la reunion habra dinero gratis"),
    ("normal",  "daremos un premio despues de la reunion"),
    ("normal", "reunion de proyecto"),
    ("normal", "proyecto para mañana"),
    ("normal", "reunion mañana"),
    ("normal", "informe del proyecto"),
    ("normal", "informe del proyecto"),
]

vocabulario = ["dinero", "gratis", "premio", "proyecto", "reunion"]

## Parte 1 — Construir el vector \(X\)

1. Escribe una función `vectorizar(texto, vocabulario)` que transforme un correo en un vector binario.
2. Vectoriza todos los correos del corpus.
3. Verifica manualmente al menos dos ejemplos.

Ejemplo esperado:


$\text{"gana dinero gratis"}\longrightarrow(1,1,0,0,0)$


In [2]:
# TODO: implementa vectorizar(texto, vocabulario)
import numpy as np
x = np.array([1, 1, 1, 1, 1])

#esto estaba en el texto, pero no creo que tenga algo interesante modificar esto.... 
#pero podemos comparar aciertos para ver con cuantos coincide jeje

In [3]:
def vectorizar(texto, vocabulario):
    palabras = texto.lower().split() #EL LOWER ES PARA EVITAR PROBLEMAS CON LAS MAYUSCULAS EN DADO CASO
    x = [1 if t in palabras else 0 for t in vocabulario]
    return x
#vectorizar(corpus,vocabulario)

#forma corda de escribir un FOR

X = [vectorizar(texto, vocabulario) for etiqueta, texto in corpus]  #aca separa la tupla en texto y etiqueta
Y = [1 if etiqueta == "spam" else 0 for etiqueta, texto in corpus]  #aca si usamos el valor de etiqueta pa algo


In [4]:
X

[[1, 1, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [0, 0, 1, 0, 1],
 [0, 0, 0, 1, 1],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 1, 0]]

In [5]:
Y


[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

## Parte 2 — Calcular el prior \(P(Y)\)

Calcula:


$P(Y=\text{spam}),\qquad P(Y=\text{normal}).$


Recuerda:

$
P(Y=y)=\frac{\#\text{correos de clase }y}{\#\text{correos totales}}.
$

In [6]:
# TODO: calcula P(Y=spam) y P(Y=normal)
n_total = len(Y)  #este para ver cuantos correos hay
n_spam = sum(Y)          # la suma de esto me dice cuantos hay con spam
n_normal = n_total - n_spam # asi, esto me dice cuandos hay sin spam y luego solo es aplicar la formula jeje

P_spam = n_spam / n_total
P_normal = n_normal / n_total

P_spam , P_normal

(0.45454545454545453, 0.5454545454545454)

## Parte 3 — Calcular $P(X_i=1\mid Y)$

Para cada palabra calcula su frecuencia dentro de cada clase. Por ejemplo:

$
P(X_{dinero}=1\mid Y=spam)
=\frac{\#\text{spam que contienen dinero}}{\#\text{spam}}.
$

Construye una tabla con las cinco palabras y ambas clases.

**Pregunta:** ¿qué ocurre si una palabra nunca aparece en una clase?

In [7]:
# TODO: calcula las probabilidades condicionales sin suavizado

def prob_condicional(X, Y, clase, vocabulario):  #Y,X fueron creadas arriba y son 2 clases, spam y normal
    # Filtramos los vectores que pertenecen a esa clase
    X_clase = [x for x, y in zip(X, Y) if y == clase] #zip produce una tupla vector / etiqueta  para cada uno
    n_clase = len(X_clase)
    #print(X_clase,"putavida") #prueba para ver si partia bien jejeje
    probs = []
    for i in range(len(vocabulario)):
        conteo = sum(x[i] for x in X_clase)  # cuántos correos de esta clase tienen la palabra i
        probs.append(conteo / n_clase)
    return probs

P_dado_spam = prob_condicional(X, Y, 1, vocabulario)
P_dado_normal = prob_condicional(X, Y, 0, vocabulario)

P_dado_spam , P_dado_normal

([0.8, 0.8, 0.4, 0.0, 0.2],
 [0.0, 0.0, 0.16666666666666666, 0.6666666666666666, 0.5])

## SI UNA PALABRA NUNCA APARECE EN UNA CLASE, ENTONCES EL MODELO CONCLUIRA CON 100% CERTEZA QUE CUANDO APAREZCA ESA PALABRA, ENTONCES SERA DE LA CLASE QUE LO TENGA, POR EJEMPLO CON LA PALABRA "DINERO", QUE SOLO APARECE EN SPAM, PUEDE LUEGO PARA LA CLASIFICACION HABER UN PROBLEMA SI APARECE EN UN CORREO NORMAL..... ENTONCES POR ESO ES BUENO TENER UNA MAYOR BASE DA DATOS !

## Parte 4 — Suavizado de Laplace

Para evitar probabilidades exactamente iguales a cero, usa suavizado de Laplace para variables Bernoulli:


$\hat P(X_i=1\mid Y=y)=\frac{N_{iy}+1}{N_y+2},$


donde \(N_{iy}\) es el número de correos de clase \(y\) que contienen la palabra \(i\), y \(N_y\) es el número total de correos de esa clase.

Calcula de nuevo la tabla.

In [8]:
# TODO: calcula probabilidades condicionales con Laplace

def prob_condicional_laplace(X, Y, clase, vocabulario):
    X_clase = [x for x, y in zip(X, Y) if y == clase]
    n_clase = len(X_clase)
    
    probs = []
    for i in range(len(vocabulario)):
        conteo = sum(x[i] for x in X_clase)
        probs.append((conteo + 1) / (n_clase + 2))   # <-- el solo se agrega en esto al parecer, algo simple..
    return probs

P_dado_spam_laplace = prob_condicional_laplace(X, Y, 1, vocabulario)
P_dado_normal_laplace = prob_condicional_laplace(X, Y, 0, vocabulario)

P_dado_spam_laplace , P_dado_normal_laplace

([0.7142857142857143,
  0.7142857142857143,
  0.42857142857142855,
  0.14285714285714285,
  0.2857142857142857],
 [0.125, 0.125, 0.25, 0.625, 0.5])

## Parte 5 — Clasificar un correo nuevo

Clasifica:

> **"dinero gratis"**

Su vector es

$x=(1,1,0,0,0).$

Bajo Naive Bayes Bernoulli:

$P(x\mid Y=y)=\prod_iP(X_i=x_i\mid Y=y).$

Recuerda que para una palabra ausente:

$P(X_i=0\mid Y=y)=1-P(X_i=1\mid Y=y).$


Calcula los scores conjuntos:

$S_y=P(Y=y)P(x\mid Y=y),$

y finalmente:

$P(Y=y\mid x)=\frac{S_y}{S_{spam}+S_{normal}}.$

Decide la clase del correo.

In [12]:
# TODO: como computar likelihood, score conjunto y posterior para "dinero gratis"

def likelihood(x, probs):
    """
    x: vector binario del correo a clasificar
    probs: lista de P(X_i=1 | Y=clase), una por palabra (salida de prob_condicional_laplace)
    """
    p = 1
    for xi, pi in zip(x, probs):
        if xi == 1:
            p *= pi
        else:
            p *= (1 - pi)
    return p

# Vectorizamos el correo nuevo
x_nuevo = vectorizar("dinero gratis", vocabulario)  # (1,1,0,0,0)

# Likelihood P(x | Y=spam) y P(x | Y=normal)
L_spam   = likelihood(x_nuevo, P_dado_spam_laplace)
L_normal = likelihood(x_nuevo, P_dado_normal_laplace)

# Scores conjuntos S_y = P(Y=y) * P(x | Y=y)
S_spam   = P_spam   * L_spam
S_normal = P_normal * L_normal

# Posterior P(Y=y | x) = S_y / (S_spam + S_normal)
P_post_spam   = S_spam   / (S_spam + S_normal)
P_post_normal = S_normal / (S_spam + S_normal)

print("L_spam:", L_spam, " L_normal:", L_normal)
print("S_spam:", S_spam, " S_normal:", S_normal)
print("P(spam|x):", P_post_spam, " P(normal|x):", P_post_normal)

clase_predicha = "spam" if P_post_spam > P_post_normal else "normal"
print("Clasificación:", clase_predicha)

L_spam: 0.17849705479859584  L_normal: 0.002197265625
S_spam: 0.08113502490845266  S_normal: 0.0011985085227272725
P(spam|x): 0.9854432517009721  P(normal|x): 0.014556748299027745
Clasificación: spam


## Parte 6 — Interpretación

Responde brevemente:

1. ¿Dónde se usa la hipótesis de independencia condicional?
2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los \(2^5\) vectores posibles?
3. ¿Por qué Naive Bayes se considera un modelo **generativo** aunque aquí lo usemos para clasificar?

## Parte 6 — Interpretación

**1. ¿Dónde se usa la hipótesis de independencia condicional?**

Se usa al calcular el *likelihood* $P(x\mid Y=y)$. En vez de calcular la probabilidad conjunta exacta de que las 5 palabras aparezcan (o no) juntas, asumimos que, una vez que ya sabemos la clase $y$, cada palabra $X_i$ es independiente de las demás. Esto es lo que permite multiplicar los 5 factores por separado (dinero, gratis, premio, proyecto, reunion) en vez de necesitar una probabilidad conjunta para toda la combinación.


**2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los $2^5$ vectores posibles?**

Sin la hipótesis de independencia, tendríamos que estimar $P(X=x\mid Y=y)$ para cada una de las $2^5=32$ combinaciones posibles de palabras, por cada una de las 2 clases — es decir, $2\cdot(2^5-1)=62$ parámetros libres. Con un corpus de solo 11 correos no habría datos suficientes para estimar casi ninguna de esas combinaciones.

Gracias a la independencia condicional, en vez de eso solo necesitamos estimar $P(X_i=1\mid Y=y)$ para cada palabra por separado:

$$
5 \text{ palabras} \times 2 \text{ clases} = 10 \text{ parámetros}
$$

Esto reduce drásticamente la cantidad de parámetros a estimar, permitiendo que un corpus pequeño sea suficiente.

---

**3. ¿Por qué Naive Bayes se considera un modelo generativo aunque aquí lo usemos para clasificar?**

Es generativo porque, en vez de modelar directamente $P(Y\mid X)$ (la frontera de decisión, como haría un modelo discriminativo como la regresión logística), modelamos cómo se "genera" cada clase: primero estimamos

$$
P(Y) \quad \text{y} \quad P(X\mid Y)
$$

es decir, qué tan probable es cada clase y cómo se comportan las palabras dentro de cada clase. Con esos dos ingredientes, en principio podríamos incluso "generar" correos artificiales típicos de spam o normal, muestreando palabras según esas probabilidades.

Para clasificar, aplicamos el teorema de Bayes al final:

$$
P(Y\mid X)\propto P(Y)\,P(X\mid Y)
$$

pero el modelo no aprendió directamente la frontera entre clases — aprendió cómo luce cada clase por separado, y de ahí deriva la clasificación.

### FIN